In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [35]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [36]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [37]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [38]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1808844357728958
epoch:  1, loss: 0.10348927974700928
epoch:  2, loss: 0.06821908801794052
epoch:  3, loss: 0.05110708996653557
epoch:  4, loss: 0.043163739144802094
epoch:  5, loss: 0.039595186710357666
epoch:  6, loss: 0.038026146590709686
epoch:  7, loss: 0.03734186291694641
epoch:  8, loss: 0.03703943267464638
epoch:  9, loss: 0.03689560294151306
epoch:  10, loss: 0.036811791360378265
epoch:  11, loss: 0.036505136638879776
epoch:  12, loss: 0.036330509930849075
epoch:  13, loss: 0.03623573109507561
epoch:  14, loss: 0.036175962537527084
epoch:  15, loss: 0.03605258837342262
epoch:  16, loss: 0.03594677522778511
epoch:  17, loss: 0.03590032830834389
epoch:  18, loss: 0.035896625369787216
epoch:  19, loss: 0.035765595734119415
epoch:  20, loss: 0.03571110963821411
epoch:  21, loss: 0.0357002317905426
epoch:  22, loss: 0.03556337207555771
epoch:  23, loss: 0.035505786538124084
epoch:  24, loss: 0.03547497093677521
epoch:  25, loss: 0.03535321727395058
epoch:  26, lo

In [39]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6944781541824341
Test metrics:  R2 = 0.6793546676635742


In [40]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    lr_init="scaled",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.11426910012960434
epoch:  1, loss: 0.07876380532979965
epoch:  2, loss: 0.06013448163866997
epoch:  3, loss: 0.05018657073378563
epoch:  4, loss: 0.04479929804801941
epoch:  5, loss: 0.041840218007564545
epoch:  6, loss: 0.04020186886191368
epoch:  7, loss: 0.03928685933351517
epoch:  8, loss: 0.038775213062763214
epoch:  9, loss: 0.038479745388031006
epoch:  10, loss: 0.03830527514219284
epoch:  11, loss: 0.03825189918279648
epoch:  12, loss: 0.038071244955062866
epoch:  13, loss: 0.03796243667602539
epoch:  14, loss: 0.03789452090859413
epoch:  15, loss: 0.03784001246094704
epoch:  16, loss: 0.037762150168418884
epoch:  17, loss: 0.03773615509271622
epoch:  18, loss: 0.03764370456337929
epoch:  19, loss: 0.0375865176320076
epoch:  20, loss: 0.03754225745797157
epoch:  21, loss: 0.03747515007853508
epoch:  22, loss: 0.037454549223184586
epoch:  23, loss: 0.03737586736679077
epoch:  24, loss: 0.037327490746974945
epoch:  25, loss: 0.03730270639061928
epoch:  26, loss

In [41]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7610422372817993
Test metrics:  R2 = 0.7262721061706543


In [42]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=10,
    lr_init="BB1",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5824660658836365
epoch:  1, loss: 0.32811781764030457
epoch:  2, loss: 0.19931748509407043
epoch:  3, loss: 0.1293788105249405
epoch:  4, loss: 0.08961616456508636
epoch:  5, loss: 0.06705516576766968
epoch:  6, loss: 0.05430849269032478
epoch:  7, loss: 0.0470421202480793
epoch:  8, loss: 0.0428827740252018
epoch:  9, loss: 0.04049604386091232
epoch:  10, loss: 0.03912870213389397
epoch:  11, loss: 0.0383475199341774
epoch:  12, loss: 0.03790176659822464
epoch:  13, loss: 0.03764752298593521
epoch:  14, loss: 0.03750230744481087
epoch:  15, loss: 0.03741905465722084
epoch:  16, loss: 0.0373709537088871
epoch:  17, loss: 0.03734273463487625
epoch:  18, loss: 0.037325792014598846
epoch:  19, loss: 0.037315234541893005
epoch:  20, loss: 0.03730612248182297
epoch:  21, loss: 0.03729358687996864
epoch:  22, loss: 0.03728800266981125
epoch:  23, loss: 0.03727291151881218
epoch:  24, loss: 0.03727121278643608
epoch:  25, loss: 0.03725285083055496
epoch:  26, loss: 0.037241

In [43]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.726125955581665
Test metrics:  R2 = 0.7196857929229736


In [44]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=10,
    lr_init="BB2",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.11301067471504211
epoch:  1, loss: 0.08345378935337067
epoch:  2, loss: 0.06537067890167236
epoch:  3, loss: 0.05432787910103798
epoch:  4, loss: 0.04759972169995308
epoch:  5, loss: 0.04350981116294861
epoch:  6, loss: 0.04102887958288193
epoch:  7, loss: 0.03952670469880104
epoch:  8, loss: 0.038618531078100204
epoch:  9, loss: 0.03807011619210243
epoch:  10, loss: 0.03773923218250275
epoch:  11, loss: 0.037539683282375336
epoch:  12, loss: 0.03741934150457382
epoch:  13, loss: 0.037346720695495605
epoch:  14, loss: 0.03730282559990883
epoch:  15, loss: 0.037276215851306915
epoch:  16, loss: 0.03726000338792801
epoch:  17, loss: 0.03725004196166992
epoch:  18, loss: 0.03724384307861328
epoch:  19, loss: 0.03724320977926254
epoch:  20, loss: 0.037237320095300674
epoch:  21, loss: 0.03723623976111412
epoch:  22, loss: 0.03723062202334404
epoch:  23, loss: 0.03722872957587242
epoch:  24, loss: 0.03722866624593735
epoch:  25, loss: 0.03722049295902252
epoch:  26, loss:

In [45]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8157200217247009
Test metrics:  R2 = 0.7671723365783691


In [46]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    lr_init="quadratic",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2891765236854553
epoch:  1, loss: 0.14133088290691376
epoch:  2, loss: 0.0777275562286377
epoch:  3, loss: 0.05307724326848984
epoch:  4, loss: 0.04319591447710991
epoch:  5, loss: 0.03927566111087799
epoch:  6, loss: 0.03775942325592041
epoch:  7, loss: 0.0371711291372776
epoch:  8, loss: 0.03692612424492836
epoch:  9, loss: 0.03680383414030075
epoch:  10, loss: 0.036574240773916245
epoch:  11, loss: 0.036390431225299835
epoch:  12, loss: 0.036317046731710434
epoch:  13, loss: 0.03631562739610672
epoch:  14, loss: 0.036116015166044235
epoch:  15, loss: 0.03603532165288925
epoch:  16, loss: 0.03599321469664574
epoch:  17, loss: 0.03584837540984154
epoch:  18, loss: 0.035755548626184464
epoch:  19, loss: 0.035710301250219345
epoch:  20, loss: 0.03557498753070831
epoch:  21, loss: 0.03547220677137375
epoch:  22, loss: 0.0354238785803318
epoch:  23, loss: 0.03529370576143265
epoch:  24, loss: 0.03517983481287956
epoch:  25, loss: 0.03512779623270035
epoch:  26, loss: 0.

In [47]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8134479522705078
Test metrics:  R2 = 0.7759608030319214


In [48]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    lr_init="lipschitz",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.06457452476024628
epoch:  1, loss: 0.05005582794547081
epoch:  2, loss: 0.043177489191293716
epoch:  3, loss: 0.039959024637937546
epoch:  4, loss: 0.03846671059727669
epoch:  5, loss: 0.037777580320835114
epoch:  6, loss: 0.0374596007168293
epoch:  7, loss: 0.0373116210103035
epoch:  8, loss: 0.03724105283617973
epoch:  9, loss: 0.03720555454492569
epoch:  10, loss: 0.037185847759246826
epoch:  11, loss: 0.037164755165576935
epoch:  12, loss: 0.037126488983631134
epoch:  13, loss: 0.03710547462105751
epoch:  14, loss: 0.037081290036439896
epoch:  15, loss: 0.03704079985618591
epoch:  16, loss: 0.03701820597052574
epoch:  17, loss: 0.03699032589793205
epoch:  18, loss: 0.03694657236337662
epoch:  19, loss: 0.03692224621772766
epoch:  20, loss: 0.03689277917146683
epoch:  21, loss: 0.03684625402092934
epoch:  22, loss: 0.03682049736380577
epoch:  23, loss: 0.03679210692644119
epoch:  24, loss: 0.03674252703785896
epoch:  25, loss: 0.0367155522108078
epoch:  26, loss: 

In [49]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.773577868938446
Test metrics:  R2 = 0.7517876625061035


In [50]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    lr_init="keep",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.41007116436958313
epoch:  1, loss: 0.24699220061302185
epoch:  2, loss: 0.15433591604232788
epoch:  3, loss: 0.10163597762584686
epoch:  4, loss: 0.0720238983631134
epoch:  5, loss: 0.055694930255413055
epoch:  6, loss: 0.04687352105975151
epoch:  7, loss: 0.042196959257125854
epoch:  8, loss: 0.039774999022483826
epoch:  9, loss: 0.038537826389074326
epoch:  10, loss: 0.03788980469107628
epoch:  11, loss: 0.03754793852567673
epoch:  12, loss: 0.03736795485019684
epoch:  13, loss: 0.037273045629262924
epoch:  14, loss: 0.03722275421023369
epoch:  15, loss: 0.037196047604084015
epoch:  16, loss: 0.0371818020939827
epoch:  17, loss: 0.0371740460395813
epoch:  18, loss: 0.0371696874499321
epoch:  19, loss: 0.037167105823755264
epoch:  20, loss: 0.03716575354337692
epoch:  21, loss: 0.03716187924146652
epoch:  22, loss: 0.03715952858328819
epoch:  23, loss: 0.03715721517801285
epoch:  24, loss: 0.03715386241674423
epoch:  25, loss: 0.037151824682950974
epoch:  26, loss: 

In [51]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7437459230422974
Test metrics:  R2 = 0.7116721868515015
